# Tokenization and Document Distance

## Introduction to Tokenization

### Description of Tokenization
- Tokenization is the process of converting raw text (doucment) into a sequence of tokens
- The sequence can be used as input to sequential models or to compute numerical features (usually token frequencies) that represents document
- Tokens: words, word pieces, or n-grams for use in natural language processing
- Input of tokenization: 
    - A set of documents (each document is a text file, e.g. an atricle), $D$
- Tokenization converts document into:
    - A sequence, $W_i$, containing a list of unique tokens in document $i$
        - Serves as inputs to sequential models
        - Keeps word order in documents, essential for LLMs
    - Or a document-term matrix, $X$, containing statistics about token frequencies in all documents
        - Each row in the matrix is a vector of statistics about token frequencies in document $i$
        - Does not keep word order

### Tokens
- Tokens are the most basic unit of representation in a text
- Tokens can be the form of:
    - Characters: documents as sequence of individual letters {h,e,l,l,o, ,w,o,r,l,d}
    - Words: documents as sequence of words {hello, world}
    - N-grams: documents as sequence of word combinations: “Study at Cambridge University" -> {"study", "at", "cambridge_university”}

### Goals of Tokenization
To summarize, the goal of tokenization is to convert documents (text) into features that are:
- Predictive in the learning task
- Interpretable by human investigators
- Easy to work with

### A Standard Pipeline of Tokenization
1. Download webpage, strip HTML, convert into a list of documents (raw text)
2. Tokenize the document at the word level, only keep words of interest as tokens
3. Normalize (lower case) the words, and build the vocabulary based on sorted unique tokens

## Pre-processing Text

### Goal of Pre-processing
- Before tokenization, text pre-processing is usually applied to raw text to remove uninformative data or shape the text into correct format
    - Uninformative data add noise and reduce statistical precision; they are also computationally costly
- Pre-processing choices can affect down-stream results, especially in unsupervised learning tasks
- Therefore, the steps of pre-processing are dependent on:
    - The purpose of the NLP task
    - The model intended to adopt

### Common Pre-processing Steps:
1. Segmenting paragraphs/sentences
2. Removing capitalization
3. Removing punctuation
4. Removing numbers
5. Dropping stopwords
6. Stemming/lemmatizing

### Segmenting Paragraphs/Sentences
- Many tasks should be done on sentences level, rather than text file as a whole
    - The python library spaCy does a good job of splitting sentences, while accounting for periods on abbreviations
- Similarily, some tasks may require document to be pharagraph rather than whole text file
    - However, there is not a grammar-based paragraph tokenizer
    - Most corpora have new paragraphs annotated
    - Or we can use line breaks to split paragraphs

### Capitalization
- Capitalization non-informative case: 
    - Capitalized/non-capitalized version of a word are equivalent
    - For example: words showing up capitalized at beginning of sentence
    - In this case, we can remove capitilization
    - In turn, removing capitalization is a standard corpus normalization technique
- Capitalization informative case: 
    - “the first amendment” versus “the First Amendment”
    - Compromise: include capitalized version of words not at beginning of sentence
- For some tasks, capitalization is important, even if capitalized/non-capitalized version of a word are equivalent:
    - Capitalization is needed for sentence splitting, part-of-speech tagging, named entity recognition, syntactic/semantic parsing
    - For sequential model (LLMs), to generate believable text, need to keep everything (include capitalization, punctuation, numbers, etc.)

### Punctuation
- Inclusion or exlcusion of punctuation depends on the task:
    - If we are vectorizing the document as a bag of words or bag of n-grams, punctuation won’t be needed
    - However, punctuation is needed for annotations (sentence splitting, parts of speech, syntax, roles, etc) or for text generation LLMs

### Numbers
- For bag of words/phrases:
    - Drop numbers, or replace with a special character (e.g. #)
- For large language models:
    - Just treat them like letters

### Stopwords
- Stopwords are common words that do not have strong semantic meaning, such as "the" and "a"
- For bag of words/phrases, we usually remove stop words:
    - However, stop words can have semantic meaning in phases and specifically legal pahses: “not guilty”, “with all deliberate speed”, etc.
    - A solution is drop stopwords by themselves, but keep them as part of phrases
    - We can filter out words and phrases using part-of-speech tags
- For LLMs, we usually keep stop words

### Stemming/lemmatizing
- Both are techniques to map multiple related word forms to a single base form (effectivly reduce dimension of vocabulary with little loss of information)
- Stemming: drop suffixes of words, e.g. "studies" -> "studi", crude but fast
- Lemmatization: convert words to their base form (returns real dictionary words), e.g. "studies" -> "study", accurate but slow
    - Although lemmatizer produces real words, applying N-grams on these words won’t make grammatical sense
    - Example: “judges have been ruling” is a sensible phrase, but after lemmatization, it would become “judge have be rule”
    - So do not use lemmatization / stemming if N-grams would be later applied (do not use when syntax matters) or need to train LLMs

## Counts and Frequencies

### Bag-of-words Representation
- Say we want to convert a corpus D to a matrix X
- In the “bag-of-words” representation, a row of X is just the frequency distribution over tokens in the document corresponding to that row
    - Tokens can be any informative features: e.g. words, n-grams, syntax features, etc
- Most basic term frequency (element in the bag-of-word matrix):
    - Term frequency of token w in document d = Count of w in document d / Total number of tokens in document d
- Other metrics can be derived through term-frequency bag-of-word matrix:
    - Document counts: number of documents where a token appears (number of rows where a specified column elements is not 0)
    - Term counts: number of total appearances of a token in corpus (sum of elements across a specified column of the matrix X)

### Application of Term Frequency: Ranking Partisan language
- Monroe et al (2009) systematically explores a number of methods for identifying words that are distinctive of groups of speakers
- Specifically, words that are distinctive of whether U.S. congressmen are Republicans or Democrats
- First, they separate speeches by topic using latent dirichlet allocation
- They then compute the difference between frequency of a term in Democrats speech and its frequency in Republicans speech for each topic
- They then ranked terms based on this frequency difference
- For example, in the topic of Abortion, the words with highest absolute difference in frequency of mentioning between Democrats and Republicans are:
    - "Women" and "Right" for Domocrats
    - "Babi" and "Abort" for Republicans

### Building a Vocabulary before Bag-of-Words
- Before computing the Bag-of_words matrix, an important step is to determine a vocabulary of words
- If we include all words in the corpus, the column dimension of the matrix will be too large (and very sparse)
- Hence, we should first inspect low-frequency words and determine a minimum document threshold to determine whether a word would be included in the vocabulary (counted by the matrix)
    - Example threshold: a word appear in at least 10 documents or 25% of documents
    - Can also impose more complex thresholds, e.g. appears twice in at least 20 documents, appears in at least 3 documents in at least 5 years

### TF-IDF Weighting
- Despite basic term frequency, the element in bag-of-words matrix can also be TF-IDF weighting of a word in the document
- TF-IDF: “Term-Frequency divided by Inverse-Document-Frequency”
- The formula for word $w$ in document $k$: 
$$\frac{\text{Count of } w \text{ in } k}{\text{Total word count of } k}
\times
\log\left(
\frac{\text{Number of documents in } D}
{\text{Count of documents containing } w}
\right)
$$
- So a word that appears frequently in a document but also appears widely in the corpus will have alower weighting
- On the other hand, the formula up-weights relatively rare words that do not appear in all documents:
    - These words are probably more distinctive of topics or differences between documents
    - This helps to pin down the important word features that are representative of a document

### Other Word Counts / Frequency
- Kelly et al (2019) suggest that including indicators for whether a phrase appears in a document (rather than the count) is often independently predictive
- We could also add log counts, quadratics in counts, etc
- It is also possible to add pairwise interactions between word counts/frequencies
- These often are not done much because of the dimensionality problem:
    - But it is potentially possible to use feature selection or principal components to help deal with that

## N-grams

### What are N-grams
- N-grams are sequences of words up to length N, however, they are not necessarily phrases / collocations
    - Consider a sentence: "I am fine"
    - unigram: one word, {"I", "am", "fine"}
    - bigram: two words, {"I am",  "am fine"}
    - trigram: three words, {"I am fine"}
- Using N-grams as token will blow up feature space:
    - Feature space dimensionality increases expotentially as we increase length N (e.g. bigram face a feature space < $V^2$ if the number of unique words in the corpus is $V$)
    - Hence, we need to filter out uninformative n-grams when forming our vocabulary
    - Usually a vocabulary formed by 10K most frequent N-grams will work well
    - For supervised learning tasks, a decent baseline is to build a vocabulary of N-grams of 60K, then use feature selection to get down to 10K
- Google Developers recommend tf-idf-weighted bigrams as a baseline representation of document for text classification tasks
    - This representation is deal for fewer, longer documents

### Why N-grams
- Conceptually, the goal of including n-grams is to featurize collocations
- Collocations > phrases: phrases are grammer structure, while collocations are phrases that people tend to use frequently with a specific meaning
- We do not want to include all phrases as vocabulary, as some of them may not contain specific meaning
- Collocations are :
    - Non-compositional: the meaning is not the sum of the parts (kick+the+bucket not the same as ”kick the bucket”)
    - Non-substitutable: cannot substitute components with synonyms (“fast food" not equal to ”quick food”, though both are phrases)
    - Non-modifiable: cannot modify with additional words or grammar (“kick around the bucket” not equal to “kick the buckets”)
- The reduction methods so far (include N-grams and set frequency threshold to determien vocabulary) do not help identify collocations (and even phrases)

###  Identifying Collocations through N-grams
- A metric for identifying collocations is point-wise mutual information:
    $$\mathrm{PMI}(w_1, w_2) = \frac{\Pr(w_1, w_2)} {\Pr(w_1)\Pr(w_2)}$$
    - that is, PMI = (Prob. of collocationn, actual) / (Prob. of collocation, if independent)
    - where w1 and w2 are words in the vocabulary, and w1_w2 is the bigram "w1 w2"
    - the metric ranks words by how often they collocate, relative to how often they occur apart
- We can then set a threshold based on this metric, and only keep bigrams exceeding this threshold as vocabulary (remove other bigrams and include unigrams instead)
- Generalizes to longer phrases (length N) as the geometric mean of the probabilities: 
    $$\mathrm{PMI}(w_1, w_2, ..., w_N) = \frac{\Pr(w_1, \dots, w_N)}{\left( \prod_{i=1}^{N} \Pr(w_i) \right)^{1/N}}$$
- Limitation: rare words that appear together once or twice will have high PMI
    - Address this with PMi thresholds + minimum frequency thresholds when pinning down vocabulary

## More Related to Tokenization

### Parts of Speech Tags
- Parts of speech (POS) tags provide useful word categories corresponding to their functions in sentences
- There are 8 main tags:
    - verb (VB), noun (NN), pronoun (PR), adjective (JJ), adverb (RB), determinant (DT), preposition (IN), conjunction (CC)
- Parts of speech tags vary in their informativeness for various tasks:
    - For categorizing topics, nouns are usually most important (spaCy can do fast noun phrase chunking)
    - For sentiment, adjectives are usually most important
- How to tuilize POS tags:
    - Can count parts of speech tags as features: e.g., using more adjectives, or using more passive verbs
    - POS n-gam frequencies are good stylistic features for authorship detection (not biased by topics/content) 
    - Example of POS bigram: first attach POS tags to words, then construct bigram to derive "noun–noun", "noun–noun", etc, and account frequencies of these bigrams

### Named Entity Recognition
- Refers to the task of identifying named entities such as “Cambridge University” as "Cambridge_University_organization" and “Marie Curie” as "Marie_Curie_person", which can be used as tokens
- Detecting the type requires a trained model (e.g. spaCy)

### Out-of-vocab Words
-  After choosing a vocabulary for featurizing a document, what to do with the words that get dropped out?
- Options:
    - Drop them
    - Replace with “unknown” token
    - Replace with part-of-speech tag
    - Replace with in-vocab hypernym (from WordNet)

## Document Distance and Clustering

### Introduction to Document Similarity
- In economics, we often want to compare documents to one another; for example, how close is a political speech to the party leader's?
- Text Re-Use algorithms (“Smith-Waterman” or BLAST) measure similarity by finding and counting shared sequences in two texts above some minimum length
- They are precise but slow (but useful for plagiarism detection)
- A more efficient approach relies on document-term matrix (bag-of-word representations)

### Cosine Similarity
- In the document-term matrix X, each document/row $X[d,:]$ is a distribution over terms in the document
    - These vectors have a spatial interpretation: geometric distances between document vectors reflect semantic distances between documents in terms of shared words
- Each column $X[:,w]$ in the document-term matrix is a distribution of the word over documents
    - These vectors also have a spatial interpretation: the geometric distances between word vectors reflect semantic distances between words in terms of showing up in the same documents
- Cosine similarity: measures similarity between document i and j by the cosine of the angle between document vector $x_i$ and $x_j$
    - With perfectly collinear documents (that is, $x_i = αx_j$ , $α > 0$), $cos(0) = 1$
    - For orthogonal documents (no words in common), $cos(π/2)=0$
- Cosine similarity is computable as the normalized dot product between the vectors: 
    $$\mathrm{cos\_sim}(x_1, x_2) = \frac{x_1 \cdot x_2}{\|x_1\| \, \|x_2\|}$$
- For a corpus with $n$ rows, the pairwise similarities give $n ×(n −1)$ similarity scores
- Applying cosine similarity on tf-idf document matrix is the workhorse: tf-idf down-weights terms that appear in many documents, usually gives better results
- Alternative distance metrics:
    - dot product and Euclidean distance (too sensitive to document length)
    - Jensen-Shannon Divergence and Jaccard distance

### K-Means Clustering
- K-means clustering: divide document vectors x into sets $S_1,...,S_k$ with associated centroids $µ_1,...,µ_k$; each doc is in set with closest centroid
- Algorithm:
    - Initialize cluster centroids randomly, each x belongs to the Set with closest centroid
    - Then centroid shift around to minimize sum of within-cluster squared distance (features should be standardized)
    - Within-cluster squared distance $$\arg\min_{S} \sum_{i=1}^{k} \sum_{x \in S_i} \left\| x - \mu_i \right\|^2$$
- k (number of clusters) is the hyperparameter of the model, can select using elbow appraoch: 
    - That is, try different k, plot the average distance within cluster (inerita) by k, find the point such that steepest decline in inerita stops 

### Other Clustering Algorithms
- K-medoid clustering:
    - Use L1 distance (summing absolut distance) rather than Euclidean distance
    - Produces the “medoid” (median vector) for each cluster rather than “centroid” (mean vector)
    - Less sensitive to outliers
- DBSCAN
    - Find non-linear clusters (which K-means and K-medoid cannot)
    - Defines clusters as continuous regions of high density
    - Detects and excludes outliers automatically
- Agglomerative clustering:
    - Makes nested clusters (hierarchical clusters)

### Word Vector
- Remember that in document-term matrix, each column $X[:,w]$ , the word vector, is a distribution over documents
- The same methods we used on the document vector can be used on the word vector:
    - Apply cosine similarity to the columns to compare words
    - Apply k-means clustering to the columns to get clusters of similar words